# 03 -- Data pipeline

`qwenmoe/data/` fetches & caches a raw corpus, and turns a flat token
stream into fixed-length `(x, y)` windows for next-token-prediction
training, plus a `make_fixed_slice` helper for compute-matched ablation
comparisons (notebook 05).

*Part of the [training-my-moe code tour](00_overview.ipynb) -- see also: [config](01_config.ipynb) - [tokenizer](02_tokenizer.ipynb) - [data](03_data.ipynb) - [model](04_model.ipynb) - [train](05_train.ipynb) - [viz](06_viz.ipynb).*

In [1]:
from pathlib import Path
import os

# Walk up until we find pyproject.toml, so these cells work no matter where Jupyter's cwd starts.
repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
os.chdir(repo_root)
print("Working directory:", Path.cwd())

Working directory: /Users/sjoseph/Documents/Study/training-my-moe


## `download_corpus` / `prepared_data_dir` (`download.py`)
The only network-touching module in the project. This repo already ran
`scripts/prepare_data.py` once, so the cell below is an instant cache hit,
not a real download.

In [2]:
from qwenmoe.config import DataConfig
from qwenmoe.data.download import download_corpus, prepared_data_dir

data_config = DataConfig(corpus="tiny_shakespeare", tokenizer_type="char")
raw_path = download_corpus(data_config.corpus, data_config.cache_dir)
print("raw corpus file:", raw_path, f"({raw_path.stat().st_size:,} bytes)")

print("prepared_data_dir (char):       ", prepared_data_dir(data_config))
print("prepared_data_dir (bpe, 4096):  ",
      prepared_data_dir(DataConfig(corpus="tiny_shakespeare", tokenizer_type="bpe", bpe_vocab_size=4096)))

raw corpus file: .cache/data/tiny_shakespeare.txt (1,115,394 bytes)
prepared_data_dir (char):        .cache/data/tiny_shakespeare_char
prepared_data_dir (bpe, 4096):   .cache/data/tiny_shakespeare_bpe4096


## `TokenDataset` -- sliding (x, y) windows (`dataset.py`)

In [3]:
import torch
from qwenmoe.data.dataset import TokenDataset

all_tokens = torch.load(".cache/data/tiny_shakespeare_char/tokens.pt", weights_only=True)
print("total cached tokens:", all_tokens.numel())

dataset = TokenDataset(all_tokens, seq_len=8)
print("num (x, y) windows:", len(dataset))

x, y = dataset[0]
print("x:", x.tolist())
print("y:", y.tolist(), " <- same span shifted right by exactly one token")
assert torch.equal(x[1:], y[:-1])
print("confirmed: y really is x shifted by one position")

total cached tokens: 1115394
num (x, y) windows: 1115386
x: [18, 47, 56, 57, 58, 1, 15, 47]
y: [47, 56, 57, 58, 1, 15, 47, 58]  <- same span shifted right by exactly one token
confirmed: y really is x shifted by one position


## `train_val_split`

In [4]:
from qwenmoe.data.dataset import train_val_split

train_tokens, val_tokens = train_val_split(all_tokens, val_fraction=0.1)
print(f"train: {train_tokens.numel():,} tokens, val: {val_tokens.numel():,} tokens "
      f"({val_tokens.numel() / all_tokens.numel():.1%} of total)")
assert torch.equal(torch.cat([train_tokens, val_tokens]), all_tokens)
print("confirmed: train+val are a contiguous, lossless split of the full stream")

train: 1,003,855 tokens, val: 111,539 tokens (10.0% of total)
confirmed: train+val are a contiguous, lossless split of the full stream


## `make_fixed_slice` -- reproducible, compute-matched slices
Splits the stream into fixed-size chunks, shuffles the chunk *order* with
a given seed, then truncates to `n_tokens`. The same `(all_tokens,
n_tokens, seed)` always returns the identical slice -- essential so every
ablation variant in a comparison run trains on exactly the same data.

In [5]:
from qwenmoe.data.dataset import make_fixed_slice

slice_a = make_fixed_slice(all_tokens, n_tokens=5000, seed=0)
slice_b = make_fixed_slice(all_tokens, n_tokens=5000, seed=0)
slice_c = make_fixed_slice(all_tokens, n_tokens=5000, seed=1)

print("same seed  -> identical slice:  ", torch.equal(slice_a, slice_b))
print("diff. seed -> different slice:  ", not torch.equal(slice_a, slice_c))
print("both still exactly n_tokens long:", slice_a.numel(), slice_c.numel())

same seed  -> identical slice:   True
diff. seed -> different slice:   True
both still exactly n_tokens long: 5000 5000


## Wiring it together: a real `DataLoader` batch

In [6]:
from torch.utils.data import DataLoader

loader = DataLoader(TokenDataset(train_tokens, seq_len=32), batch_size=4, shuffle=True)
x_batch, y_batch = next(iter(loader))
print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)

x_batch shape: torch.Size([4, 32])
y_batch shape: torch.Size([4, 32])
